# P0 · Second referee report: random-sample baseline, token norms, near-duplicates (exploratory)

| Section | Question |
|---|---|
| Random sample of real patches, with AUPRO (MVTec AD, VisA, 3CAD) | how much of the codebook-vs-coreset gap is "k-means is good" and how much is "greedy k-center is a poor choice at small M"? |
| Token norms (DINOv2-B, with and without registers, MVTec AD) | do the highest-scoring patches of normal images have unusually large token norms, as the artifact-token explanation of the K = 64 collapse predicts? |
| Near-duplicates (3CAD) | does a perceptual hash find training images that nearly match test images, beyond exact copies? |

MVTec AD and VisA reuse cached features (no download). 3CAD needs its archive in `MyDrive/bovw-forensics/data/`
and re-extracts features. **Runtime (T4):** about 1 h, plus 3–4 h for the 3CAD random run (last section).
Finished runs are saved; Run all again after a disconnect to resume.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Random-sample baseline on MVTec AD and VisA

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
from bovw.sweep import load_config
from bovw import anomaly
for ds in ["mvtec", "visa"]:
    anomaly.run(load_config(f"configs/p0_random_{ds}.yaml"))

In [ ]:
import pandas as pd
from pathlib import Path
from bovw import stats
from bovw.sweep import load_results

RES = "/content/drive/MyDrive/bovw-forensics/results"
BASE = {"mvtec": "p0_anomaly_mvtec_v2", "visa": "p0_anomaly_visa", "3cad": "p0_anomaly_3cad"}

def random_comparison(ds):
    base = pd.read_csv(f"{RES}/{BASE[ds]}.csv")
    base = base[base.method.isin(["codebook_dist", "coreset_knn"]) & (base.extractor == "dinov2_s")]
    rnd = load_results(Path(RES) / f"p0_random_{ds}.jsonl")
    d = pd.concat([base, rnd], ignore_index=True)
    d["k"] = d["k"].astype(int)
    m = ["aupro", "pixel_auroc", "image_auroc", "image_auroc_top10"]
    out = pd.concat([stats.compare({ds: d}, a, b, ks=(64, 256, 1024), metrics=m)
                     for a, b in [("codebook_dist", "random_knn"), ("random_knn", "coreset_knn")]])
    means = d.groupby(["method", "k", "category"])[m].mean().groupby(level=[0, 1]).mean().round(3)
    return out, means

tabs = []
for ds in ["mvtec", "visa"]:
    out, means = random_comparison(ds)
    out.to_csv(f"{RES}/p0_random_{ds}_paired.csv", index=False)
    means.to_csv(f"{RES}/p0_random_{ds}_means.csv")
    display(means)
    tabs.append(out)
pd.concat(tabs).drop(columns=["p_sign", "p_wilcoxon"]).round(3)

## 3 · Token norms of the highest-scoring normal patches (DINOv2-B, MVTec AD, K = 64, seed 0)

In [ ]:
from bovw import ad_data, review
CACHE = "/content/drive/MyDrive/bovw-forensics/cache"
rows = []
for ex in ["dinov2_b", "dinov2_b_reg"]:
    for cat in ad_data.MVTEC_CATEGORIES:
        try:
            rows.append({"category": cat, **review.token_norm_check(CACHE, f"mvtec_{cat}_s448_m256", ex, k=64)})
        except Exception as e:
            print(ex, cat, "skipped:", e)
tn = pd.DataFrame(rows)
tn.to_csv(f"{RES}/p0_token_norms_mvtec.csv", index=False)
cols = ["codebook_top_patch_norm_pct_median", "codebook_top_patch_in_top1pct_norm",
        "coreset_top_patch_norm_pct_median", "coreset_top_patch_in_top1pct_norm",
        "spearman_codebook_score_vs_norm", "spearman_coreset_score_vs_norm"]
tn.groupby("extractor")[cols].mean().round(3)

## 4 · 3CAD: perceptual near-duplicates, then the random-sample baseline

In [ ]:
import torch
import os, glob, shutil, subprocess
from bovw import ad_data

def get_mvtec():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz", "/content/mvtec"
    if not os.path.isfile(tar):
        raise SystemExit("Put mvtec_anomaly_detection.tar.xz in MyDrive/bovw-forensics/data/ (see notebook 02).")
    if not os.path.isdir(f"{local}/bottle"):
        os.makedirs(local, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return local

def get_visa():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar", "/content/visa"
    if not os.path.isfile(tar) or os.path.getsize(tar) < 1e9:
        ad_data.ensure_archive(tar, url=ad_data.VISA_URL, sha256=None)
    if not os.path.isdir(local):
        os.makedirs(local)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return ad_data.find_visa_root(local)

def get_3cad():
    local, archive = "/content/3cad", "/content/3cad_archive"
    if not os.path.isdir(local):
        copy = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
        if not copy:
            raise SystemExit("Put the 3CAD archive in MyDrive/bovw-forensics/data/ (see notebook 06).")
        shutil.copy(copy[0], archive)
        os.makedirs(local)
        magic = open(archive, "rb").read(4)
        fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
        shutil.unpack_archive(archive, local, fmt)
        os.remove(archive)
    return ad_data.find_root(local, "Copper_Stator")

ROOT3 = get_3cad()
nd = pd.DataFrame([review.near_duplicates(ROOT3, c, thresholds=(0, 4, 8)) for c in ad_data.THREECAD_CATEGORIES])
nd.to_csv(f"{RES}/p0_3cad_near_duplicates.csv", index=False)
nd

In [ ]:
cfg = load_config("configs/p0_random_3cad.yaml")
cfg["data"]["root"] = ROOT3
anomaly.run(cfg)
out, means = random_comparison("3cad")
out.to_csv(f"{RES}/p0_random_3cad_paired.csv", index=False)
means.to_csv(f"{RES}/p0_random_3cad_means.csv")
display(means)
out.drop(columns=["p_sign", "p_wilcoxon"]).round(3)